In [6]:
import json
import importlib
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

In [ ]:
#prosta klasa do wyboru najlepszego modelu (tutaj na podstawie accuracy) 
class ModelSelector:
    def __init__(self, models_path):
        with open(models_path, 'r') as f:
            self.models_config = json.load(f)
        self.best_model_pipeline = None
        self.best_score = -1

    def _get_clf_class(self, class_string):
        module_name, class_name = class_string.rsplit('.', 1)
        module = importlib.import_module(module_name)
        return getattr(module, class_name)

    def _create_preprocessing_pipeline(self, X):
        numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
        categorical_features = X.select_dtypes(include=['object', 'bool', 'category']).columns

        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])

        return ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features)
            ])

    def fit(self, X, y):
        preprocessor = self._create_preprocessing_pipeline(X)
        
        for i, model_entry in enumerate(self.models_config):
            if i == 1:
                print(f" Model {model_entry['name']} został pominięty.")
                continue

            name = model_entry['name']
            clf_class = self._get_clf_class(model_entry['class'])
            params = model_entry['params'].copy()
             
            #niektore parametry ulegly zmianie w nowszych wersjach sklearn, dlatego wprowadzam poprawki
            # HistGradientBoosting
            if 'HistGradientBoosting' in model_entry['class'] and params.get('loss') == 'auto':
                params['loss'] = 'log_loss'
            
            #Modele drzewiaste (RandomForest, ExtraTrees, DecisionTree)
            tree_based_models = ['RandomForest', 'ExtraTrees', 'DecisionTree']
            if any(tree_model in model_entry['class'] for tree_model in tree_based_models):
                # Usuwamy min_impurity_split (wycofany)
                if 'min_impurity_split' in params:
                    del params['min_impurity_split']
                
                # Usuwamy tol (nie istnieje w tych modelach)
                if 'tol' in params:
                    del params['tol']
                
                # Zamieniamy 'auto' na 'sqrt' dla max_features
                if params.get('max_features') == 'auto':
                    params['max_features'] = 'sqrt'

            clf = clf_class(**params)
            
            current_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', clf)
            ])
            
            try:
                scores = cross_val_score(current_pipeline, X, y, cv=5, scoring='accuracy', n_jobs= 1)
                mean_score = np.mean(scores)
                
                print(f" {i+1}/{len(self.models_config)} Model: {name} | Średnie Accuracy: {mean_score:.4f}")
                
                if mean_score > self.best_score:
                    self.best_score = mean_score
                    self.best_model_pipeline = current_pipeline
            except Exception as e:
                print(f"Błąd przy modelu {name}: {e}")

        if self.best_model_pipeline:
            self.best_model_pipeline.fit(X, y)
            print(f"\nZAKOŃCZONO. Wybrano najlepszy model: {self.best_model_pipeline.named_steps['classifier'] } ze średnim accuracy: {self.best_score:.4f}")
        else:
            raise ValueError("Nie udało się wytrenować żadnego modelu.")

    def predict(self, X):
        if self.best_model_pipeline is None:
            raise ValueError("Model nie został jeszcze dopasowany.")
        return self.best_model_pipeline.predict(X)

In [38]:
X = pd.read_csv('X.csv')
y = pd.read_csv('y.csv').values.ravel()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

selector = ModelSelector('models.json')
selector.fit(X_train, y_train)
#docelowo chyba usuniemy komunikaty o poszegolnych modelach

 1/35 Model: kr_vs_kp_HistGradientBoostingClassifier | Średnie Accuracy: 0.5453
 Model breast_w_SVC został pominięty.
 3/35 Model: credit_approval_HistGradientBoostingClassifier | Średnie Accuracy: 0.5794
 4/35 Model: credit_g_RandomForestClassifier | Średnie Accuracy: 0.5945
 5/35 Model: diabetes_RandomForestClassifier | Średnie Accuracy: 0.5844
 6/35 Model: spambase_RandomForestClassifier | Średnie Accuracy: 0.5744
 7/35 Model: tic_tac_toe_RandomForestClassifier | Średnie Accuracy: 0.5848
 8/35 Model: electricity_HistGradientBoostingClassifier | Średnie Accuracy: 0.5715
 9/35 Model: sick_RandomForestClassifier | Średnie Accuracy: 0.5808
 10/35 Model: pc4_RandomForestClassifier | Średnie Accuracy: 0.5808
 11/35 Model: pc3_RandomForestClassifier | Średnie Accuracy: 0.5959
 12/35 Model: jm1_HistGradientBoostingClassifier | Średnie Accuracy: 0.5708
 13/35 Model: kc2_RandomForestClassifier | Średnie Accuracy: 0.5905
 14/35 Model: kc1_RandomForestClassifier | Średnie Accuracy: 0.5700
 15/3

In [39]:
selector.predict(X_test)
accuracy = np.mean(selector.predict(X_test) == y_test)
print(f"\nDokładność na zbiorze testowym: {accuracy:.4f}")


Dokładność na zbiorze testowym: 0.5825


In [40]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')

In [41]:
#Sprawdzenie na prostszych danych
X = titanic.data
y = titanic.target
y = y.astype(int).astype(bool)
print("Rozmiar danych:", X.shape)

Rozmiar danych: (1309, 13)


In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
selector = ModelSelector('models.json')
selector.fit(X_train, y_train)

 1/35 Model: kr_vs_kp_HistGradientBoostingClassifier | Średnie Accuracy: 0.9465
 Model breast_w_SVC został pominięty.
 3/35 Model: credit_approval_HistGradientBoostingClassifier | Średnie Accuracy: 0.9542
 4/35 Model: credit_g_RandomForestClassifier | Średnie Accuracy: 0.9551
 5/35 Model: diabetes_RandomForestClassifier | Średnie Accuracy: 0.9542
 6/35 Model: spambase_RandomForestClassifier | Średnie Accuracy: 0.9513
 7/35 Model: tic_tac_toe_RandomForestClassifier | Średnie Accuracy: 0.9513
 8/35 Model: electricity_HistGradientBoostingClassifier | Średnie Accuracy: 0.9513
 9/35 Model: sick_RandomForestClassifier | Średnie Accuracy: 0.9513
 10/35 Model: pc4_RandomForestClassifier | Średnie Accuracy: 0.9484
 11/35 Model: pc3_RandomForestClassifier | Średnie Accuracy: 0.9523
 12/35 Model: jm1_HistGradientBoostingClassifier | Średnie Accuracy: 0.9446
 13/35 Model: kc2_RandomForestClassifier | Średnie Accuracy: 0.9542
 14/35 Model: kc1_RandomForestClassifier | Średnie Accuracy: 0.9503
 15/3

In [43]:
selector.predict(X_test)
accuracy = np.mean(selector.predict(X_test) == y_test)
print(f"\nDokładność na zbiorze testowym: {accuracy:.4f}")


Dokładność na zbiorze testowym: 0.9580


In [7]:
from sklearn.ensemble import VotingClassifier
from operator import itemgetter

class ModelSelector:
    def __init__(self, models_path):
        with open(models_path, 'r') as f:
            self.models_config = json.load(f)
        self.best_model_pipeline = None
        self.best_score = -1
        self.results = []

    def _get_clf_class(self, class_string):
        module_name, class_name = class_string.rsplit('.', 1)
        module = importlib.import_module(module_name)
        return getattr(module, class_name)

    def _create_preprocessing_pipeline(self, X):
        numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
        categorical_features = X.select_dtypes(include=['object', 'bool', 'category']).columns

        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])

        return ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features)
            ])

    def fit(self, X, y):
        preprocessor = self._create_preprocessing_pipeline(X)
        self.results = [] 
        
        for i, model_entry in enumerate(self.models_config):
            if i == 1: #na razie pomijamy bo nie jestem w stanie przeliczyć 
                print(f" Model {model_entry['name']} został pominięty.")
                continue

            name = model_entry['name']
            clf_class = self._get_clf_class(model_entry['class'])
            params = model_entry['params'].copy()
             
            # Poprawki parametrów
            if 'HistGradientBoosting' in model_entry['class'] and params.get('loss') == 'auto':
                params['loss'] = 'log_loss'
            
            tree_based_models = ['RandomForest', 'ExtraTrees', 'DecisionTree']
            if any(tree_model in model_entry['class'] for tree_model in tree_based_models):
                if 'min_impurity_split' in params: del params['min_impurity_split']
                if 'tol' in params: del params['tol']
                if params.get('max_features') == 'auto': params['max_features'] = 'sqrt'

            clf = clf_class(**params)
            
            current_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', clf)
            ])
            
            try:
                scores = cross_val_score(current_pipeline, X, y, cv=5, scoring='balanced_accuracy', n_jobs=1)
                mean_score = np.mean(scores)
                
                self.results.append({
                    'name': name,
                    'score': mean_score,
                    'clf': clf 
                })
                
                print(f" {i+1}/{len(self.models_config)} Model: {name} | Mean balanced accuracy: {mean_score:.4f}")
                
            except Exception as e:
                print(f"Błąd przy modelu {name}: {e}")

        if not self.results:
            raise ValueError("Nie udało się wytrenować żadnego modelu.")

        self.results.sort(key=itemgetter('score'), reverse=True)
        best = self.results[0]
        
        top_5_results = self.results[:5]
        estimators = [(res['name'], res['clf']) for res in top_5_results]
        voting_clf = VotingClassifier(estimators=estimators, voting='hard')
        
        voting_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', voting_clf)
        ])

        print(f"\n Porównanie VotingClassifier z najlepszym pojedynczym modelem")
        
        try:
            v_scores = cross_val_score(voting_pipeline, X, y, cv=5, scoring='balanced_accuracy', n_jobs=1)
            v_score = np.mean(v_scores)
            
            print(f" Średnie Balanced Accuracy (Voting): {v_score:.4f}")
            print(f" Średnie Balanced Accuracy dla ({best['name']}): {best['score']:.4f}")

            if v_score > best['score']:
                print("Wybrano VotingClassifier składający się z modeli:")
                for name, _ in estimators:
                    print(f" - {name}")
                self.best_model_pipeline = voting_pipeline
                self.best_score = v_score
            else:
                print(f"Wybrano pojedynczy model: {best['name']}")
                self.best_model_pipeline = Pipeline(steps=[
                    ('preprocessor', preprocessor),
                    ('classifier', best['clf'])
                ])
                self.best_score = best['score']

        except Exception as e:
            print(f" Błąd przy budowaniu VotingClassifier: {e}. Wybieramy dotychczas najlepszy model.")
            self.best_model_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', best['clf'])
            ])
            self.best_score = best['score']

        self.best_model_pipeline.fit(X, y)

    def predict(self, X):
        if self.best_model_pipeline is None:
            raise ValueError("Model nie został jeszcze dopasowany.")
        return self.best_model_pipeline.predict(X)

In [8]:
X = pd.read_csv('X.csv')
y = pd.read_csv('y.csv').values.ravel()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

selector = ModelSelector('models.json')
selector.fit(X_train, y_train)

 1/35 Model: kr_vs_kp_HistGradientBoostingClassifier | Mean balanced accuracy: 0.5308
 Model breast_w_SVC został pominięty.
 3/35 Model: credit_approval_HistGradientBoostingClassifier | Mean balanced accuracy: 0.5606
 4/35 Model: credit_g_RandomForestClassifier | Mean balanced accuracy: 0.5520
 5/35 Model: diabetes_RandomForestClassifier | Mean balanced accuracy: 0.5619
 6/35 Model: spambase_RandomForestClassifier | Mean balanced accuracy: 0.5541
 7/35 Model: tic_tac_toe_RandomForestClassifier | Mean balanced accuracy: 0.5492
 8/35 Model: electricity_HistGradientBoostingClassifier | Mean balanced accuracy: 0.5552
 9/35 Model: sick_RandomForestClassifier | Mean balanced accuracy: 0.5565
 10/35 Model: pc4_RandomForestClassifier | Mean balanced accuracy: 0.5470
 11/35 Model: pc3_RandomForestClassifier | Mean balanced accuracy: 0.5522
 12/35 Model: jm1_HistGradientBoostingClassifier | Mean balanced accuracy: 0.5538
 13/35 Model: kc2_RandomForestClassifier | Mean balanced accuracy: 0.5552
 

In [9]:
balanced_accuracy = balanced_accuracy_score(y_test, selector.predict(X_test))
print(f"\nBalanced Accuracy na zbiorze testowym: {balanced_accuracy:.4f}")


Balanced Accuracy na zbiorze testowym: 0.5571
